# Hardware y Escala — Módulo 15

Este notebook es el compañero práctico del módulo de arquitectura. Aquí vas a:

1. Inspeccionar el hardware de tu propia máquina desde Python
2. Ver la diferencia de vectorización con tus propios ojos (y temporizador)
3. Observar efectos de caché en código real
4. Medir FLOPs empíricamente en multiplicaciones de matrices
5. Derivar presupuestos de memoria con fórmulas reproducibles
6. Estimar trabajo de entrenamiento sin inventar precios ni rendimiento

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/15_arquitectura_de_computadoras/code/01_arquitectura.ipynb)

In [ ]:
%pip install numpy matplotlib psutil -q

In [ ]:
import platform
import psutil
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # para compatibilidad en Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print(f"Python:    {platform.python_version()}")
print(f"NumPy:     {np.__version__}")
print(f"Plataforma: {platform.system()} {platform.machine()}")

---
## Sección 1: Tu hardware

Python puede interrogar al sistema operativo sobre el hardware disponible.
Esto es útil para entender en qué entorno corre tu código y qué puedes esperar de él.

In [ ]:
# Información de CPU
print("=== CPU ===")
print(f"Arquitectura:       {platform.machine()}")
print(f"Procesador:         {platform.processor() or 'N/D (ver /proc/cpuinfo)'}")
print(f"Núcleos físicos:    {psutil.cpu_count(logical=False)}")
print(f"Núcleos lógicos:    {psutil.cpu_count(logical=True)}")
freq = psutil.cpu_freq()
if freq:
    print(f"Frecuencia actual:  {freq.current:.0f} MHz")
    print(f"Frecuencia máx:     {freq.max:.0f} MHz")

In [ ]:
# Información de memoria y almacenamiento
print("=== Memoria ===")
ram = psutil.virtual_memory()
print(f"RAM total:          {ram.total / 2**30:.1f} GiB")
print(f"RAM disponible:     {ram.available / 2**30:.1f} GiB")
print(f"RAM en uso:         {ram.percent:.1f}%")

print("\n=== Almacenamiento ===")
for part in psutil.disk_partitions()[:2]:
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"{part.mountpoint}: {usage.total / 2**30:.0f} GiB total, "
              f"{usage.free / 2**30:.0f} GiB libres")
    except PermissionError:
        pass

In [ ]:
# ¿Hay GPU disponible?
print("=== GPU ===")
try:
    import torch
    if torch.cuda.is_available():
        print(f"CUDA disponible: SÍ")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory
        print(f"VRAM: {mem / 2**30:.1f} GiB")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("Metal (Apple GPU): disponible")
    else:
        print("Sin GPU acelerada disponible — usando CPU")
except ImportError:
    print("PyTorch no instalado — no se puede verificar GPU")
    print("En Colab: Runtime > Change runtime type > GPU para activarla")

> **💡 Prueba esto:** Compara tus resultados con los de alguien más en la clase.
> ¿Cuántos núcleos físicos vs lógicos tiene cada máquina? ¿Cuánta RAM?
> En Colab (CPU runtime): ¿cuántos núcleos obtienes? ¿Cuánto RAM?

---
## Sección 2: Vectorización — ver la diferencia

Esta es la demostración más importante del notebook.
Vamos a hacer exactamente la misma operación de tres formas y medir el tiempo.

In [ ]:
N = 2_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
referencias_listas = len(a) + len(b)
print(f"Listas Python: {referencias_listas:,} referencias a enteros en 2 contenedores")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

In [ ]:
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")

In [ ]:
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"

In [ ]:
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nEn esta corrida, NumPy fue {speedup:.0f} veces más rápido.")
print("La suma de NumPy es una ufunc compilada de CPU sobre memoria contigua.")
print("Evita el despacho de Python por elemento y puede usar SIMD; esta prueba no demuestra BLAS ni varios threads.")

> **💡 Prueba esto:**
> - Cambia `N` a 100_000 y a 100_000_000. ¿Cómo cambia el speedup?
> - Prueba con `np.float32` en vez de `np.float64`. ¿Es más rápido?
> - ¿Qué pasa si haces `a_np * b_np` (multiplicación) en vez de suma?

---
## Sección 3: Efectos de caché

Acceder a datos contiguos suele aprovechar mejor las líneas de caché que recorrerlos
con saltos. La magnitud, e incluso el orden de los tiempos, depende de la máquina.

Esto no es sobre Python — es sobre física del hardware.

In [ ]:
# Creamos una matriz cuadrada grande
SIZE = 2000
M = np.random.randn(SIZE, SIZE).astype(np.float32)
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")

In [ ]:
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")

In [ ]:
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].sum()  # leer columna completa — saltos en memoria
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

### ¿Por qué ocurre esto?

Cuando NumPy lee `M[i, :]` (una fila), los datos están contiguos en memoria —
el procesador los carga en líneas de caché que contienen varios elementos.
Las lecturas cercanas pueden reutilizar bytes ya transferidos.

Cuando lee `M[:, j]` (una columna), cada elemento está a `SIZE × 4 = 8,000 bytes`
del anterior. El patrón aprovecha menos bytes de cada línea y puede causar más fallos de caché;
no implica que cada acceso falle ni que todos lleguen hasta RAM.

```
Fila [i, :]:    [0][1][2][3][4]...  ← contiguos, reutilización probable
Columna [:, j]: [0]....[1]....[2].. ← separados, más tráfico potencial
```

En pandas: `.iterrows()` accede por filas a un DataFrame que internamente
almacena columnas. Por eso es especialmente lento.

> **💡 Prueba esto:**
> - Cambia `SIZE` a 200. ¿Qué cambia? No asumas que cabe en L3: consulta tu CPU.
> - Prueba con `SIZE = 5000`. ¿Se amplifica el efecto?
> - ¿Qué pasa si conviertes la columna a array contiguo antes: `M[:, j].copy().sum()`?

---
## Sección 4: Multiplicación de matrices y FLOPs

La multiplicación de matrices aparece en redes neuronales y ciencia de datos. Vamos a medirla
y estimar su tasa con la convención de trabajo $2N^3$ para una matriz cuadrada.

In [ ]:
def benchmark_matmul(N, reps=3):
    """Mide tiempo de N×N matmul y calcula GFLOPS observados."""
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)

    # Warm-up
    _ = A @ B

    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        C = A @ B
        times.append(time.perf_counter() - t0)

    t_med = np.median(times)
    flops = 2 * N**3           # DERIVED: convención aproximada de trabajo GEMM
    gflops = flops / t_med / 1e9
    return t_med, gflops

sizes = [64, 128, 256, 512, 1024, 2048]
print(f"{'N':>6}  {'Tiempo (ms)':>12}  {'GFLOPS obs.':>12}")
print("-" * 36)
results = []
for N in sizes:
    t, g = benchmark_matmul(N)
    results.append((N, t, g))
    print(f"{N:>6}  {t*1000:>12.2f}  {g:>12.1f}")

In [ ]:
# Graficar GFLOPS observados vs N
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), facecolor="#1a1a2e")

for ax in [ax1, ax2]:
    ax.set_facecolor("#16213e")
    for sp in ax.spines.values():
        sp.set_edgecolor("#2a2a4e")
    ax.tick_params(colors="#e0e0e0")
    ax.xaxis.label.set_color("#e0e0e0")
    ax.yaxis.label.set_color("#e0e0e0")
    ax.title.set_color("#e0e0e0")
    ax.yaxis.grid(True, color="#2a2a4e", alpha=0.5)
    ax.set_axisbelow(True)

ns     = [r[0] for r in results]
times  = [r[1] * 1000 for r in results]  # ms
gflops = [r[2] for r in results]

ax1.plot(ns, times, 'o-', color="#0db7ed", linewidth=2, markersize=7)
ax1.set_xlabel("Tamaño de matriz N")
ax1.set_ylabel("Tiempo (ms)")
ax1.set_title("Tiempo de N×N matmul")

ax2.plot(ns, gflops, 's-', color="#f0a500", linewidth=2, markersize=7)
ax2.set_xlabel("Tamaño de matriz N")
ax2.set_ylabel("GFLOPS observados")
ax2.set_title("GFLOPS observados según N")

fig.suptitle("Multiplicación de matrices: tiempo y GFLOPS",
             color="#e0e0e0", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("\nNumPy ejecuta A @ B en CPU y normalmente delega la operación a una biblioteca BLAS.")
print("BLAS puede usar varios threads, según la instalación y sus variables de entorno.")
print("Los GFLOPS no tienen que crecer de forma monótona: influyen caché, threads, kernels y ruido.")

> **💡 Prueba esto:**
> - Compara `np.float32` vs `np.float64`. ¿Cuánto cambia el rendimiento?
> - Si tienes GPU disponible: prueba con `torch.matmul` en CUDA y compara.
> - ¿Qué GFLOPS teóricos tiene tu CPU? (Busca el modelo y sus specs.)

---
## Sección 5: Presupuesto reproducible de memoria

**FACT (convención decimal):** en esta sección, mil millones = $10^9$, GB = $10^9$ bytes
y TB = $10^{12}$ bytes. GiB es una unidad binaria distinta.

**DERIVED:** la memoria de pesos es parámetros × bits / 8. La cuenta no incluye
caché KV, activaciones, gradientes, estados del optimizador ni temporales.

Fuente técnica: [Hugging Face — Anatomy of Model's Memory](https://huggingface.co/docs/transformers/model_memory_anatomy).

In [ ]:
def memoria_pesos_gb(parametros_miles_millones: float, bits: int) -> float:
    """Devuelve GB decimales ocupados sólo por los pesos."""
    return parametros_miles_millones * bits / 8

modelos_memoria = [7, 70, 1_000]  # miles de millones de parámetros
precisiones = [("BF16", 16), ("INT8", 8), ("INT4", 4)]

print(f"{'Parámetros':>24}  {'BF16 (GB)':>10}  {'INT8 (GB)':>10}  {'INT4 (GB)':>10}")
print("-" * 62)
for parametros in modelos_memoria:
    memorias = [memoria_pesos_gb(parametros, bits) for _, bits in precisiones]
    etiqueta = f"{parametros:,} mil millones"
    print(f"{etiqueta:>24}  {memorias[0]:>10,.1f}  {memorias[1]:>10,.1f}  {memorias[2]:>10,.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.set_axisbelow(True)

x = np.arange(len(modelos_memoria))
ancho = 0.24
for i, (nombre, bits) in enumerate(precisiones):
    valores = [memoria_pesos_gb(p, bits) for p in modelos_memoria]
    ax.bar(x + (i - 1) * ancho, valores, width=ancho, label=nombre)
ax.set_yscale("log")
ax.set_xticks(x, ["7 mil millones", "70 mil millones", "1 millón de millones"])
ax.set_ylabel("GB decimales, sólo pesos")
ax.set_title("DERIVED — Memoria de pesos por precisión", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

> **💡 Prueba esto:**
> - Agrega un tamaño de 13 mil millones de parámetros.
> - Calcula cuánto cambia sólo el presupuesto de pesos entre BF16 e INT4.
> - Compara GB decimales con la capacidad que tu sistema reportó en GiB; no mezcles unidades.

---
## Sección 6: Trabajo de entrenamiento como escenario

**ESTIMATE (modelo de cálculo):** para un Transformer denso, una aproximación docente
del trabajo de entrenamiento es $6 \times N_{parámetros} \times N_{tokens}$. Estima FLOP,
no tiempo, energía ni costo. Esos resultados requieren throughput observado del sistema completo.

Fuente primaria del modelo de escala: [Hoffmann et al., Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556).

In [ ]:
def estimar_trabajo_entrenamiento(
    parametros_miles_millones: float,
    tokens_miles_millones: float,
) -> float:
    """ESTIMATE: trabajo aproximado en FLOP para un Transformer denso."""
    parametros = parametros_miles_millones * 1e9
    tokens = tokens_miles_millones * 1e9
    return 6 * parametros * tokens

def horas_desde_throughput_medido(
    trabajo_flop: float, rendimiento_observado_sistema_tflops: float
) -> float:
    """DERIVED: duración si el throughput medido del sistema se sostiene."""
    if rendimiento_observado_sistema_tflops <= 0:
        raise ValueError("El throughput observado debe ser positivo")
    return trabajo_flop / (rendimiento_observado_sistema_tflops * 1e12) / 3600

print("Funciones listas: separan el trabajo estimado del throughput que debes medir.")

In [ ]:
escenarios_trabajo = [
    ("1 mil millones / 20 mil millones", 1, 20),
    ("7 mil millones / 140 mil millones", 7, 140),
    ("70 mil millones / 1.4 millones de millones", 70, 1_400),
]

print(f"{'Escenario (parámetros / tokens)':<48} {'ESTIMATE FLOP':>16}")
print("-" * 66)
for nombre, parametros, tokens in escenarios_trabajo:
    trabajo = estimar_trabajo_entrenamiento(parametros, tokens)
    print(f"{nombre:<48} {trabajo:>16.2e}")

print("\nPara estimar horas, introduce throughput observado del sistema completo, no pico de ficha técnica.")

In [ ]:
# DERIVED: presupuesto base de una receta mixed precision + Adam clásico
def estado_modelo_gb(parametros_miles_millones: float, bytes_por_parametro: float = 18) -> float:
    return parametros_miles_millones * bytes_por_parametro

parametros_estado = [1, 7, 70]
estado_gb = [estado_modelo_gb(p) for p in parametros_estado]
pesos_bf16_gb = [memoria_pesos_gb(p, 16) for p in parametros_estado]

fig, ax = plt.subplots(figsize=(9, 5), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.set_axisbelow(True)

x = np.arange(len(parametros_estado))
ax.bar(x - 0.18, pesos_bf16_gb, width=0.36, label="Sólo pesos BF16")
ax.bar(x + 0.18, estado_gb, width=0.36, label="Receta 18 bytes/parámetro")
ax.set_xticks(x, ["1 mil millones", "7 mil millones", "70 mil millones"])
ax.set_yscale("log")
ax.set_ylabel("GB decimales")
ax.set_title("DERIVED — Pesos frente a estado agregado del modelo", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

> **💡 Prueba esto:**
> - Llama a `estimar_trabajo_entrenamiento` con un escenario hipotético propio.
> - Mide throughput del sistema completo antes de usar `horas_desde_throughput_medido`.
> - Explica por qué un pico de ficha técnica no sustituye esa medición.

---
## Ejercicio integrador

**ESTIMATE (escenario):** quieres estudiar un Transformer denso de 7 mil millones de
parámetros con 140 mil millones de tokens. No se proporciona precio ni rendimiento: debes
separar lo derivable de lo que hace falta medir.

In [ ]:
# Tu código aquí

# 1. DERIVED: calcula GB decimales de sólo pesos BF16.

# 2. DERIVED: calcula el estado agregado con la receta de 18 bytes por parámetro.

# 3. ESTIMATE: calcula el trabajo 6 × parámetros × tokens.

# 4. Lista las mediciones necesarias para estimar duración, energía y costo.
#    Incluye throughput observado, potencia AC, duración y tarifa documentada.

# 5. Explica cómo sharding cambia memoria local y comunicación sin cambiar el estado agregado.

<details>
<summary>Derivaciones comprobables</summary>

```python
parametros = 7e9
tokens = 140e9
pesos_bf16_gb = parametros * 2 / 1e9
estado_receta_gb = parametros * 18 / 1e9
trabajo_estimado_flop = 6 * parametros * tokens
print(pesos_bf16_gb, estado_receta_gb, trabajo_estimado_flop)
```
</details>

---
## Resumen

| Experimento | Lección |
|-------------|----------|
| Inspección de hardware | Cada máquina reporta restricciones concretas: cores, RAM y aceleradores |
| Vectorización | Una ufunc compilada evita despacho Python y puede usar SIMD |
| Efectos de caché | El orden de acceso cambia localidad; el efecto se mide en cada máquina |
| MatMul y BLAS | NumPy usa CPU/BLAS; los GFLOPS observados pueden ser no monótonos |
| Memoria de modelos | Pesos son sólo una parte del estado agregado |
| Trabajo de entrenamiento | Una ESTIMATE de FLOP no es una estimación de precio ni duración |

**El mensaje central:** el hardware no es un detalle de implementación.
Es el presupuesto físico dentro del cual vive cada algoritmo.